In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV

from sklearn.preprocessing import StandardScaler

from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, roc_auc_score

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer

import joblib

In [ ]:
import pandas as pd

df = pd.read_csv(
    r"D:\Classes\IIT_Patna_Ai_ML\Capstone_Project\House_Price_Project\part1\data\train.csv"
)

df.head()

In [ ]:
df.shape

In [ ]:
median_price = df["SalePrice"].median()

df["price_class"] = (
    df["SalePrice"] >= median_price
).astype(int)


df["price_class"].value_counts()

In [ ]:
X = df.drop(
    columns=["SalePrice","price_class","Id"]
)

y = df["price_class"]


print(X.shape)
print(y.shape)

In [ ]:
X = pd.get_dummies(
    X,
    drop_first=True
)


print(X.shape)

In [ ]:
X_train, X_test, y_clf_train, y_clf_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


print(X_train.shape)
print(X_test.shape)

print(y_clf_train.shape)
print(y_clf_test.shape)

In [ ]:
scaler = StandardScaler()


X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns
)


X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns
)


print(X_train_scaled.shape)
print(X_test_scaled.shape)

In [ ]:
dt_default = DecisionTreeClassifier(
    random_state=42
)


dt_default.fit(
    X_train_scaled,
    y_clf_train
)


dt_train_accuracy = dt_default.score(
    X_train_scaled,
    y_clf_train
)


dt_test_accuracy = dt_default.score(
    X_test_scaled,
    y_clf_test
)


print("Decision Tree Baseline")
print("----------------------")
print("Training Accuracy:", dt_train_accuracy)
print("Testing Accuracy :", dt_test_accuracy)

In [ ]:
dt_controlled = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=20,
    random_state=42
)


dt_controlled.fit(
    X_train_scaled,
    y_clf_train
)


dt_control_train_accuracy = dt_controlled.score(
    X_train_scaled,
    y_clf_train
)


dt_control_test_accuracy = dt_controlled.score(
    X_test_scaled,
    y_clf_test
)


print("Controlled Decision Tree")
print("------------------------")
print("Training Accuracy:", dt_control_train_accuracy)
print("Testing Accuracy :", dt_control_test_accuracy)

In [ ]:
dt_gini = DecisionTreeClassifier(
    max_depth=5,
    criterion="gini",
    random_state=42
)


dt_entropy = DecisionTreeClassifier(
    max_depth=5,
    criterion="entropy",
    random_state=42
)


dt_gini.fit(
    X_train_scaled,
    y_clf_train
)


dt_entropy.fit(
    X_train_scaled,
    y_clf_train
)


gini_test_accuracy = dt_gini.score(
    X_test_scaled,
    y_clf_test
)


entropy_test_accuracy = dt_entropy.score(
    X_test_scaled,
    y_clf_test
)


print("Gini Test Accuracy:", gini_test_accuracy)

print("Entropy Test Accuracy:", entropy_test_accuracy)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)


rf.fit(
    X_train_scaled,
    y_clf_train
)


rf_train_accuracy = rf.score(
    X_train_scaled,
    y_clf_train
)


rf_test_accuracy = rf.score(
    X_test_scaled,
    y_clf_test
)


rf_auc = roc_auc_score(
    y_clf_test,
    rf.predict_proba(X_test_scaled)[:,1]
)


print("Random Forest")
print("----------------")
print("Training Accuracy:", rf_train_accuracy)
print("Testing Accuracy :", rf_test_accuracy)
print("ROC-AUC           :", rf_auc)

In [ ]:
feature_importance = pd.DataFrame({

    "Feature": X_train.columns,

    "Importance": rf.feature_importances_

})


feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)


feature_importance.head(5)

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")


X_train_scaled_imputed = pd.DataFrame(
    imputer.fit_transform(X_train_scaled),
    columns=X_train_scaled.columns
)


X_test_scaled_imputed = pd.DataFrame(
    imputer.transform(X_test_scaled),
    columns=X_test_scaled.columns
)


print(X_train_scaled_imputed.isnull().sum().sum())
print(X_test_scaled_imputed.isnull().sum().sum())

In [ ]:
gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)


gb.fit(
    X_train_scaled_imputed,
    y_clf_train
)


gb_train_accuracy = gb.score(
    X_train_scaled_imputed,
    y_clf_train
)


gb_test_accuracy = gb.score(
    X_test_scaled_imputed,
    y_clf_test
)


gb_auc = roc_auc_score(
    y_clf_test,
    gb.predict_proba(X_test_scaled_imputed)[:,1]
)


print("Gradient Boosting")
print("------------------")
print("Training Accuracy:", gb_train_accuracy)
print("Testing Accuracy :", gb_test_accuracy)
print("ROC-AUC           :", gb_auc)

In [ ]:
log_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)


log_model.fit(
    X_train_scaled_imputed,
    y_clf_train
)


print("Logistic Regression trained successfully")

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


models = {

    "Logistic Regression": log_model,

    "Controlled Decision Tree": dt_controlled,

    "Random Forest": rf,

    "Gradient Boosting": gb

}


cv_results = []


for name, model in models.items():

    scores = cross_val_score(
        model,
        X_train_scaled_imputed,
        y_clf_train,
        cv=cv,
        scoring="roc_auc"
    )


    cv_results.append(
        [
            name,
            scores.mean(),
            scores.std()
        ]
    )


cv_table = pd.DataFrame(
    cv_results,
    columns=[
        "Model",
        "Mean AUC",
        "Std AUC"
    ]
)


cv_table

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV

pipeline = make_pipeline(

    SimpleImputer(strategy="median"),

    StandardScaler(),

    RandomForestClassifier(
        random_state=42
    )

)

pipeline

In [ ]:
param_grid = {

    "randomforestclassifier__n_estimators": [50, 100, 200],

    "randomforestclassifier__max_depth": [5, 10, None],

    "randomforestclassifier__min_samples_leaf": [1, 5]

}

In [ ]:
grid = GridSearchCV(

    pipeline,

    param_grid,

    cv=cv,

    scoring="roc_auc",

    n_jobs=-1

)


grid.fit(
    X_train,
    y_clf_train
)


print("Best Parameters:")
print(grid.best_params_)


print("\nBest CV AUC:")
print(grid.best_score_)

In [ ]:
lowest_features = feature_importance.tail(5)

lowest_features

In [ ]:
lowest_feature_names = lowest_features["Feature"].tolist()


X_train_reduced = X_train_scaled.drop(
    columns=lowest_feature_names
)


X_test_reduced = X_test_scaled.drop(
    columns=lowest_feature_names
)


print("Original features:", X_train_scaled.shape[1])
print("Reduced features :", X_train_reduced.shape[1])

In [ ]:
rf_reduced = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42
)


rf_reduced.fit(
    X_train_reduced,
    y_clf_train
)


reduced_auc = roc_auc_score(
    y_clf_test,
    rf_reduced.predict_proba(X_test_reduced)[:,1]
)


print("Full Model AUC:", rf_auc)

print("Reduced Model AUC:", reduced_auc)

In [ ]:
best_pipeline = grid.best_estimator_


fractions = [0.2, 0.4, 0.6, 0.8, 1.0]


learning_results = []


for f in fractions:

    size = int(f * len(X_train))


    X_subset = X_train.iloc[:size]

    y_subset = y_clf_train.iloc[:size]


    best_pipeline.fit(
        X_subset,
        y_subset
    )


    train_auc = roc_auc_score(
        y_subset,
        best_pipeline.predict_proba(X_subset)[:,1]
    )


    test_auc = roc_auc_score(
        y_clf_test,
        best_pipeline.predict_proba(X_test)[:,1]
    )


    learning_results.append(
        [
            f,
            train_auc,
            test_auc
        ]
    )


learning_table = pd.DataFrame(
    learning_results,
    columns=[
        "Training Fraction",
        "Training AUC",
        "Test AUC"
    ]
)


learning_table

In [ ]:
import joblib

joblib.dump(
    best_pipeline,
    "best_model.pkl"
)

print("Model saved successfully")

In [ ]:
import os

print(os.path.exists("best_model.pkl"))

In [ ]:
import joblib
import pandas as pd


# Load saved model
loaded_model = joblib.load("best_model.pkl")


# Create two sample rows from test data
sample_rows = X_test.iloc[:2]


# Make predictions
predictions = loaded_model.predict(sample_rows)


print("Predictions:")
print(predictions)